In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
from spx_history import *

- for one symbol first

In [2]:
start_dt = dt.date(2021,1,1)
end_dt = dt.date(2026,2,1)

- combine all dividends data

In [3]:
# dates = pd.date_range(start_dt, end_dt).date
# td_l = []
# for d in dates:
# 	date_s = d.strftime("%Y-%m-%d")
# 	tdf = pd.DataFrame()
# 	try:
# 		tdf = pd.read_csv(f'data_by_date/dividends/{date_s}.csv')
# 	except Exception as e:
# 		qt.log.warning(f"file not found for date : {d}")
# 	qt.log.info(f"[{d}] tdf.shape = {tdf.shape}")
# 	td_l.append(tdf)

# td = pd.concat(td_l).sort_values(['Date', 'Code']).reset_index(drop=True)
# td.to_csv('data_by_date/all_dividends_till_20260201.csv', index=False)

- one symbol only

In [ ]:
def get_adjusted_intraday_by_sym(symbol='GOOGL', all_adj=None):
	if all_adj is None:
		all_adj = 

In [4]:
symbol = "GOOGL"
ts = get_data_by_symbol_filename(symbol=symbol, filename='split')
td = get_data_by_symbol_filename(symbol=symbol, filename='div')
ti = get_data_by_symbol_filename(symbol=symbol, filename='intraday')

# convert date to date type
ts['date'] = pd.to_datetime(ts['date']).dt.date

[JUSTY.LOG]	2026-02-18 15:24:33,071 - qt.common.help - INFO - df.shape = (1, 2)
[JUSTY.LOG]	2026-02-18 15:24:33,075 - qt.common.help - INFO - df.shape = (7, 8)
[JUSTY.LOG]	2026-02-18 15:24:33,680 - qt.common.help - INFO - df.shape = (921908, 7)


In [5]:
all_d = pd.read_csv('data_by_date/all_dividends_till_20260201.csv')
all_d['Date'] = pd.to_datetime(all_d['Date']).dt.date

In [6]:
ti = add_dt_us_intraday(ti)
ti = ti.sort_values('datetime_us').reset_index(drop=True)
ti['close'] = ti['close'].ffill()

In [8]:
ti['time'].min()
ti['time'].max()

datetime.time(4, 0)

datetime.time(19, 59)

In [9]:
start_date_sym	= ti['date'].min()
end_date_sym	= ti['date'].max()
start_time_sym	= dt.time(4, 0) # ti['time'].min()
end_time_sym	= dt.time(19, 59) # ti['time'].max()

dates = pd.bdate_range(start=start_date_sym, end=end_date_sym, freq="D").date

times = pd.date_range(
	start=dt.datetime.combine(dt.date.today(), start_time_sym),
	end=dt.datetime.combine(dt.date.today(), end_time_sym),
	freq="1min"
).time

# cartesian product
lattice = pd.MultiIndex.from_product(
	[dates, times],
	names=["date", "time"]
).to_frame(index=False)

# do a left merge
til = pd.merge(lattice, ti, on=['date', 'time'], how='left')

In [10]:
til['close'] = til.groupby('date')['close'].ffill()

In [11]:
# read holidays
th_us = pd.read_csv('us_holidays.csv')
th_us['date'] = pd.to_datetime(th_us['date'], dayfirst=True).dt.date
us_hols = th_us['date'].unique().tolist()

# keep close using prices in trading hours
til_c = pd.DataFrame(til.groupby('date')['close'].last()).reset_index().sort_values('date').reset_index(drop=True)
til_c['is_holiday'] = til_c['date'].apply(lambda x: x in us_hols)
til_c['is_weekend'] = pd.to_datetime(til_c['date']).dt.weekday >= 5

# drop weekends
til_c = til_c[~til_c['is_weekend']].sort_values('date').reset_index(drop=True)
til_c.drop(columns=['is_weekend'], inplace=True)

# forward fill close
til_c['close'] = til_c['close'].ffill()

# parse split ratio
ts["split_ratio"] = ts["split"].apply(lambda x: float(x.split("/")[1])/float(x.split("/")[0]))

# add split ratio and dividend data
til_c = pd.merge(til_c, ts[['date', 'split_ratio']], on='date', how='left')
til_c = pd.merge(til_c, all_d[all_d['Code'] == symbol].rename({'Date' : 'date', 'Dividend' : 'div_usd'}, axis=1)[['date', 'div_usd']], on='date', how='left')

# shift div and split adjustment prior to ex-date
til_c['div_usd_pre_ex'] = til_c['div_usd'].shift(-1)
til_c['div_adj'] = 1-(til_c['div_usd_pre_ex']/til_c['close'])
til_c['split_adj'] = til_c['split_ratio'].shift(-1)

# sort date
til_c = til_c.sort_values('date', ascending=False).reset_index(drop=True)

# cumulative split and div adjustments
til_c['split_adj_cum'] = til_c['split_adj'].fillna(1).cumprod()
til_c['div_adj_cum'] = til_c['div_adj'].fillna(1).cumprod()

# cumulative adjustments
til_c['adj_cum'] = til_c['split_adj_cum'] * til_c['div_adj_cum']

# adjust close
til_c['close_adj'] = til_c['close']*til_c['adj_cum']

# add log and raw returns
til_c["log_ret_1d"] = 100 * np.log(til_c["close_adj"]/til_c["close_adj"].shift(-1))
til_c["ret_1d"] = 100 * (-1 + (til_c["close_adj"]/til_c["close_adj"].shift(-1)))

In [ ]:
# add adjustment factor to the lattice
til = pd.merge(til, til_c[['date', 'adj_cum']], on='date', how='left')

# combine date and time column
til["datetime_us"] = pd.to_datetime(til["date"].astype(str) + " " + til["time"].astype(str))

# filter useful columns
til = til[['datetime_us', 'date', 'time', 'open', 'high', 'low', 'close', 'volume', 'adj_cum']]

In [30]:
qt.view(til[til['date'] == dt.date(2024, 5, 22)])

Grid(columns_fit='auto', compress_data=True, css_rules_down=['.number-cell {text-align: left;width: 10;}', '.l…

In [13]:
qt.view(til_c)

Grid(columns_fit='auto', compress_data=True, css_rules_down=['.number-cell {text-align: left;width: 10;}', '.l…